# Results Dashboard

This notebook loads all saved experiment results and generates paper-ready tables and figures.
Run this after completing all experiments (notebooks 03-07) to produce publication materials.

In [ ]:
import json
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"font.size": 12, "figure.dpi": 150})

print("Results Dashboard")

## Step 1: Load All Results

In [ ]:
results_dir = "../outputs"
result_files = sorted(glob.glob(os.path.join(results_dir, "*_results.json")))

all_results = {}
for f in result_files:
    try:
        with open(f) as fp:
            data = json.load(fp)
        name = data.get("experiment", os.path.basename(f))
        all_results[name] = data
        print(f"Loaded: {name}")
    except Exception as e:
        print(f"Failed to load {f}: {e}")

print(f"\nTotal: {len(all_results)} experiments loaded")

## Step 2: Classification Results

In [ ]:
cls_rows = []
for name, data in all_results.items():
    if data.get("task") == "classification":
        clean = data.get("clean", {})
        obf = data.get("obfuscated", {})
        cls_rows.append({
            "Experiment": name,
            "Model": data.get("model", ""),
            "Clean Acc": f"{clean.get('accuracy', 0):.4f}",
            "Obf Acc": f"{obf.get('accuracy', 0):.4f}",
            "Clean F1": f"{clean.get('f1', 0):.4f}",
            "Obf F1": f"{obf.get('f1', 0):.4f}",
        })

if cls_rows:
    df_cls = pd.DataFrame(cls_rows)
    print("Classification Results:")
    print(df_cls.to_string(index=False))
    print(f"\nLaTeX:\n{df_cls.to_latex(index=False)}")
else:
    print("No classification results found")

## Step 3: Detection Results

In [ ]:
det_rows = []
for name, data in all_results.items():
    if data.get("task") in ("object_detection", "zero_shot_object_detection"):
        clean = data.get("clean", {})
        obf = data.get("obfuscated", {})
        det_rows.append({
            "Experiment": name,
            "Clean mAP": f"{clean.get('map', 0):.4f}",
            "Obf mAP": f"{obf.get('map', 0):.4f}",
            "Clean mAP@50": f"{clean.get('map_50', 0):.4f}",
            "Obf mAP@50": f"{obf.get('map_50', 0):.4f}",
        })

if det_rows:
    df_det = pd.DataFrame(det_rows)
    print("Detection Results:")
    print(df_det.to_string(index=False))
    print(f"\nLaTeX:\n{df_det.to_latex(index=False)}")
else:
    print("No detection results found")

## Step 4: Segmentation Results

In [ ]:
seg_rows = []
for name, data in all_results.items():
    if data.get("task") in ("segmentation", "zero_shot_segmentation"):
        clean = data.get("clean", {})
        obf = data.get("obfuscated", {})
        seg_rows.append({
            "Experiment": name,
            "Clean mIoU": f"{clean.get('miou', 0):.4f}",
            "Obf mIoU": f"{obf.get('miou', 0):.4f}",
        })

if seg_rows:
    df_seg = pd.DataFrame(seg_rows)
    print("Segmentation Results:")
    print(df_seg.to_string(index=False))
    print(f"\nLaTeX:\n{df_seg.to_latex(index=False)}")
else:
    print("No segmentation results found")

## Step 5: Unified Results Overview

In [ ]:
METRIC_MAP = {
    "classification": ("accuracy", "Accuracy"),
    "object_detection": ("map", "mAP"),
    "zero_shot_object_detection": ("map", "mAP"),
    "segmentation": ("miou", "mIoU"),
    "zero_shot_segmentation": ("miou", "mIoU"),
}

unified_rows = []
for name, data in all_results.items():
    task = data.get("task", "unknown")
    metric_key, metric_label = METRIC_MAP.get(task, ("accuracy", "Accuracy"))
    clean = data.get("clean", {})
    obf = data.get("obfuscated", {})
    clean_val = clean.get(metric_key, 0)
    obf_val = obf.get(metric_key, 0)
    delta = obf_val - clean_val
    unified_rows.append({
        "Experiment": name,
        "Task": task,
        "Metric": metric_label,
        "Clean": f"{clean_val:.4f}",
        "Obfuscated": f"{obf_val:.4f}",
        "Delta": f"{delta:+.4f}",
    })

if unified_rows:
    df_unified = pd.DataFrame(unified_rows)
    print("Unified Results Overview:")
    print(df_unified.to_string(index=False))
    print(f"\nLaTeX:\n{df_unified.to_latex(index=False)}")
else:
    print("No results available")

## Step 6: Export Figures

In [ ]:
os.makedirs(os.path.join(results_dir, "figures"), exist_ok=True)

if all_results:
    names = list(all_results.keys())
    clean_vals = []
    obf_vals = []
    for name in names:
        data = all_results[name]
        clean = data.get("clean", {})
        obf = data.get("obfuscated", {})
        clean_vals.append(clean.get("accuracy", clean.get("map", clean.get("miou", 0))))
        obf_vals.append(obf.get("accuracy", obf.get("map", obf.get("miou", 0))))

    fig, ax = plt.subplots(figsize=(12, max(4, len(names) * 0.6)))
    y = range(len(names))
    height = 0.35
    ax.barh([i - height/2 for i in y], clean_vals, height, label="Clean", color="#2196F3")
    ax.barh([i + height/2 for i in y], obf_vals, height, label="Obfuscated", color="#FF9800")
    ax.set_yticks(list(y))
    ax.set_yticklabels(names)
    ax.set_xlabel("Score")
    ax.set_title("Clean vs. Obfuscated Performance")
    ax.legend()
    plt.tight_layout()
    fig.savefig(os.path.join(results_dir, "figures", "performance_comparison.pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(results_dir, "figures", "performance_comparison.png"), bbox_inches="tight")
    plt.show()
    print("Figures saved to outputs/figures/")
else:
    print("No results to plot")

## Step 7: Export All Tables

In [ ]:
tables_dir = os.path.join(results_dir, "tables")
os.makedirs(tables_dir, exist_ok=True)

for name, df in [("classification", df_cls if 'df_cls' in dir() else None),
                  ("detection", df_det if 'df_det' in dir() else None),
                  ("segmentation", df_seg if 'df_seg' in dir() else None)]:
    if df is not None and not df.empty:
        path = os.path.join(tables_dir, f"{name}_results.csv")
        df.to_csv(path, index=False)
        print(f"Saved {path}")

print("Export complete")